# Data Visualization Quiz

In [1]:
import base64
from io import BytesIO
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

_csv = base64.b64decode("...")
_raw = pd.read_csv(BytesIO(_csv))
df = _raw.rename(columns={
    'Country/Territory': 'Country',
    'Area (km²)': 'Area',
    'Density (per km²)': 'Density',
    'World Population Percentage': 'World share (%)'
})
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(
    id_vars=['CCA3', 'Country', 'Continent'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Population'
)
df_long['Year'] = df_long['Year'].str[:4].astype(int)
years = sorted(df_long['Year'].unique())

## Q1 — Which continent is growing fastest?

**Chart choice & justification:** A line chart compares each continent’s population index against its own 1970 baseline (1970 = 100), so relative growth is comparable despite very different starting populations.

In [2]:
q1 = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = q1[q1['Year'] == 1970][['Continent', 'Population']].rename(columns={'Population': 'Base'})
q1 = q1.merge(base, on='Continent')
q1['Index'] = q1['Population'] / q1['Base'] * 100
leader = q1[q1['Year'] == 2022].nlargest(1, 'Index').iloc[0]
fig = px.line(q1, x='Year', y='Index', color='Continent',
              title=f"{leader['Continent']} grew fastest: {leader['Index']-100:.1f}% above its 1970 population",
              labels={'Index': 'Population index (1970 = 100)', 'Year': ''},
              color_discrete_sequence=['#2E75B6','#E07B39','#777777','#F5C842','#6A9F58','#9467BD'],
              template='plotly_white')
fig.update_traces(line_width=2.7)
fig.update_layout(legend_title_text='', yaxis_showgrid=False, xaxis_showgrid=False,
                  margin=dict(l=25,r=25,t=75,b=25))
fig.show()

## Q2 — Which countries added the most people?

**Chart choice & justification:** A sorted horizontal bar chart makes the 15 country-level additions directly rankable; highlighting African countries makes the continent count visible without a second chart.

In [3]:
q2 = df[['Country', 'Continent', '1970 Population', '2022 Population']].copy()
q2['People added'] = q2['2022 Population'] - q2['1970 Population']
top15 = q2.nlargest(15, 'People added').sort_values('People added')
africa_count = int((top15['Continent'] == 'Africa').sum())
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other continents')
fig = px.bar(top15, x='People added', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa':'#E07B39','Other continents':'#A6A6A6'},
             text='People added',
             title=f"{africa_count} of the 15 largest population gains were in Africa",
             labels={'People added':'Additional people, 1970–2022','Country':''},
             template='plotly_white')
fig.update_traces(texttemplate='%{x:.2s}', textposition='outside', cliponaxis=False)
fig.update_layout(legend_title_text='', xaxis_showgrid=False, yaxis_showgrid=False,
                  margin=dict(l=20,r=75,t=75,b=25))
fig.show()

## Q3 — Do bigger countries have more people?

**Chart choice & justification:** A log–log scatter plot tests the area–population relationship across several orders of magnitude; continent colour and labelled density extremes reveal countries that depart from the broad pattern.

In [4]:
q3 = df.copy()
q3['Density'] = q3['2022 Population'] / q3['Area']
high_density = q3.nlargest(2, 'Density')
low_density = q3.nsmallest(2, 'Density')
fig = px.scatter(q3, x='Area', y='2022 Population', color='Continent',
                 hover_name='Country', hover_data={'Density':':.1f','Area':':,','2022 Population':':,'},
                 log_x=True, log_y=True, opacity=.72,
                 title='Larger land areas generally contain more people, but density varies sharply',
                 labels={'Area':'Land area (km², log scale)','2022 Population':'Population (log scale)'},
                 template='plotly_white',
                 color_discrete_sequence=['#2E75B6','#E07B39','#777777','#F5C842','#6A9F58','#9467BD'])
for _, r in pd.concat([high_density, low_density]).iterrows():
    fig.add_annotation(x=r['Area'], y=r['2022 Population'], text=r['Country'],
                       showarrow=True, arrowhead=2, ax=25, ay=-25, font=dict(size=10))
fig.update_layout(legend_title_text='', xaxis_showgrid=False, yaxis_showgrid=False,
                  margin=dict(l=25,r=25,t=75,b=25))
fig.show()

## Q4 — How crowded are the continents?

**Chart choice & justification:** A violin plot shows each continent’s full density distribution, including skew and concentration, rather than reducing countries to a single average.

In [5]:
q4 = df.copy()
q4['Density'] = q4['2022 Population'] / q4['Area']
order = q4.groupby('Continent')['Density'].median().sort_values().index.tolist()
fig = px.violin(q4, x='Continent', y='Density', color='Continent',
                category_orders={'Continent':order}, box=True, points='all',
                hover_name='Country',
                title='North America has the highest median density; Macau and Monaco stretch the upper tail',
                labels={'Density':'People per km²','Continent':''},
                template='plotly_white',
                color_discrete_sequence=['#A6A6A6','#2E75B6','#E07B39','#777777','#F5C842','#6A9F58'])
fig.update_traces(meanline_visible=True, jitter=.25, pointpos=0)
fig.update_layout(showlegend=False, xaxis_showgrid=False, yaxis_showgrid=False,
                  margin=dict(l=25,r=25,t=75,b=25))
fig.show()

## Q5 — Where did populations grow — and where did they shrink?

**Chart choice & justification:** A choropleth locates percentage change geographically, making regional clusters of growth and population decline easier to identify than a country ranking alone.

In [6]:
q5 = df.copy()
q5['Change (%)'] = (q5['2022 Population'] / q5['2000 Population'] - 1) * 100
q5 = q5.dropna(subset=['CCA3'])
largest_growth = q5.nlargest(1, 'Change (%)').iloc[0]
largest_decline = q5.nsmallest(1, 'Change (%)').iloc[0]
fig = px.choropleth(q5, locations='CCA3', locationmode='ISO-3', color='Change (%)',
                    hover_name='Country',
                    hover_data={'Change (%)':':.1f','Continent':True,'CCA3':False},
                    color_continuous_scale=[[0,'#2E75B6'],[0.5,'#F2F2F2'],[1,'#E07B39']],
                    color_continuous_midpoint=0,
                    title=f"Population change, 2000–2022: {largest_growth['Country']} grew most; {largest_decline['Country']} declined most",
                    labels={'Change (%)':'Population change (%)'},
                    template='plotly_white')
fig.update_layout(margin=dict(l=0,r=0,t=75,b=10), coloraxis_colorbar=dict(title='% change'))
fig.show()

## Q6 — Where do the world's 8 billion people live?

**Chart choice & justification:** A treemap encodes population by rectangle area and nests countries inside continents, showing both requested levels in one view.

In [7]:
q6 = df.groupby(['Continent','Country'], as_index=False)['2022 Population'].sum()
fig = px.treemap(q6, path=['Continent','Country'], values='2022 Population',
                 color='Continent',
                 color_discrete_sequence=['#2E75B6','#E07B39','#AAAAAA','#F5C842','#6A9F58','#9467BD'],
                 title='Asia contains the largest share of the 2022 world population',
                 template='plotly_white')
fig.update_traces(root_color='#D9D9D9', textinfo='label+percent parent',
                  hovertemplate='<b>%{label}</b><br>Population: %{value:,}<br>Share of parent: %{percentParent:.1%}<extra></extra>')
fig.update_layout(margin=dict(l=10,r=10,t=75,b=10), showlegend=False)
fig.show()

## Q7 — Whose share of the world is rising?

**Chart choice & justification:** A 100% stacked bar chart compares the three named continents’ shares with the combined remainder at each year, showing how the global population composition shifts over time.

In [8]:
q7 = df_long.groupby(['Year','Continent'], as_index=False)['Population'].sum()
world = q7.groupby('Year')['Population'].transform('sum')
q7['Share (%)'] = q7['Population'] / world * 100
q7['Group'] = np.where(q7['Continent'].isin(['Africa','Asia','Europe']), q7['Continent'], 'Rest of the world')
q7 = q7.groupby(['Year','Group'], as_index=False)['Share (%)'].sum()
group_order = ['Africa','Asia','Europe','Rest of the world']
q7['Group'] = pd.Categorical(q7['Group'], categories=group_order, ordered=True)
q7 = q7.sort_values(['Year','Group'])
africa_1970 = q7[(q7.Year==1970)&(q7.Group=='Africa')]['Share (%)'].iloc[0]
africa_2022 = q7[(q7.Year==2022)&(q7.Group=='Africa')]['Share (%)'].iloc[0]
europe_1970 = q7[(q7.Year==1970)&(q7.Group=='Europe')]['Share (%)'].iloc[0]
europe_2022 = q7[(q7.Year==2022)&(q7.Group=='Europe')]['Share (%)'].iloc[0]
fig = px.bar(q7, x='Year', y='Share (%)', color='Group', barmode='stack',
             category_orders={'Group':group_order},
             color_discrete_map={'Africa':'#E07B39','Asia':'#2E75B6','Europe':'#777777','Rest of the world':'#D9D9D9'},
             title=f"Africa's share rose {africa_2022-africa_1970:+.1f} pp; Europe's fell {europe_2022-europe_1970:+.1f} pp",
             labels={'Year':'','Share (%)':'Share of world population (%)'},
             template='plotly_white')
fig.update_layout(yaxis_range=[0,100], yaxis_showgrid=False, xaxis_showgrid=False,
                  legend_title_text='', margin=dict(l=25,r=25,t=75,b=25))
fig.show()